# 🚀 Capstone 9.6: Deploy & Predict

**Goal**: Serve the champion model as a REST API and send predictions.

---

In [ ]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np
import requests
import json
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

mlflow.autolog(disable=True)

wine = load_wine()
X = pd.DataFrame(wine.data, columns=wine.feature_names)
y = wine.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

MODEL_NAME = "wine-quality-capstone"
print("✅ Setup complete!")

## 1. Direct Prediction (Python)

In [ ]:
# Load champion model
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

# Predict
predictions = champion.predict(X_test)
acc = accuracy_score(y_test, predictions)

print(f"🏆 Champion model — test accuracy: {acc:.4f}")
print(f"\nSample predictions:")
for i in range(8):
    match = "✅" if predictions[i] == y_test[i] else "❌"
    print(f"  {match} predicted={wine.target_names[predictions[i]]}, actual={wine.target_names[y_test[i]]}")

## 2. Start the REST API Server

Open a **separate terminal** and run:

```bash
cd c:\Users\sujat\projects\MLFlow_Learn
mlflow models serve -m models:/wine-quality-capstone@champion -p 1234 --no-conda
```

Wait for: `Listening at: http://127.0.0.1:1234`

In [ ]:
# Print the exact serve command
print("📋 Copy this to your terminal:")
print(f"\nmlflow models serve -m models:/{MODEL_NAME}@champion -p 1234 --no-conda")

## 3. Send REST API Predictions

In [ ]:
SERVER_URL = "http://127.0.0.1:1234"

def predict_api(data, url=SERVER_URL):
    """Send prediction request to the model server."""
    payload = {
        "dataframe_split": {
            "columns": list(data.columns),
            "data": data.values.tolist()
        }
    }
    try:
        response = requests.post(
            f"{url}/invocations",
            headers={"Content-Type": "application/json"},
            json=payload,
            timeout=10
        )
        response.raise_for_status()
        return response.json()
    except requests.ConnectionError:
        return {"error": "Server not running. Start it with the command above."}
    except Exception as e:
        return {"error": str(e)}

# Test
result = predict_api(X_test[:5])
print(f"API Response: {result}")
if "error" not in result:
    preds = result.get("predictions", result)
    print(f"\n✅ API is working!")
    print(f"Predicted: {preds}")
    print(f"Actual:    {list(y_test[:5])}")
else:
    print(f"\n⚠️ {result['error']}")

## 4. Docker Deployment (Bonus)

Build a Docker image of the champion model:

In [ ]:
print("📋 Docker deployment commands:")
print(f"")
print(f"# 1. Build Docker image from champion model")
print(f"mlflow models build-docker -m models:/{MODEL_NAME}@champion -n capstone-wine-model")
print(f"")
print(f"# 2. Run container")
print(f"docker run -p 8080:8080 capstone-wine-model")
print(f"")
print(f"# 3. Test it (port 8080)")
print(f'curl http://localhost:8080/ping')
print(f"")
print(f"# 4. Cleanup")
print(f"docker stop $(docker ps -q --filter ancestor=capstone-wine-model)")

## 5. Batch Prediction

In [ ]:
# Batch prediction — process entire test set
champion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")

all_predictions = champion.predict(X_test)

# Create results DataFrame
results = X_test.copy()
results["predicted_class"] = all_predictions
results["predicted_name"] = [wine.target_names[p] for p in all_predictions]
results["actual_class"] = y_test
results["actual_name"] = [wine.target_names[a] for a in y_test]
results["correct"] = all_predictions == y_test

print(f"📊 Batch Predictions:")
print(f"   Total samples: {len(results)}")
print(f"   Correct: {results['correct'].sum()}")
print(f"   Wrong: {(~results['correct']).sum()}")
print(f"   Accuracy: {results['correct'].mean():.4f}")

# Show misclassified samples
wrong = results[~results['correct']]
if len(wrong) > 0:
    print(f"\n❌ Misclassified samples ({len(wrong)}):")
    print(wrong[["predicted_name", "actual_name"]].to_string())

---

## 🎓🎉 Capstone Complete!

Congratulations! You've built a complete end-to-end ML workflow with MLFlow:

- ✅ **Data preparation** with EDA logged as artifacts
- ✅ **Experiment tracking** with 5+ model types
- ✅ **Hyperparameter optimization** with Optuna + nested runs
- ✅ **Model comparison** with visualizations
- ✅ **Model Registry** with champion/challenger workflow
- ✅ **Deployment** as REST API and Docker container
- ✅ **Batch prediction** for offline scoring

### What's Next?

- Try replacing the Wine dataset with your own data
- Deploy to **AWS** using Module 7's approach
- Set up **PostgreSQL** backend for production (Module 8)
- Add **CI/CD** to automate model retraining and deployment
- Explore **MLFlow + deep learning** frameworks (PyTorch, TensorFlow)